# NULLXES SHINRA — Google Colab (A100)

Не запускать 4B веса на локальной RTX 2080.
Этот ноутбук: клон репо, зависимости, tokenizer DNA, meta param count.
Полный pretrain: 8× A100 80GB, `configs/pretrain_a100.yaml`.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → A100"
name = torch.cuda.get_device_name(0)
print(name)
assert "2080" not in name, "RTX 2080 is not a SHINRA training target"

In [ ]:
!git clone https://github.com/MagistrTheOne/NULLXES-SHINRA-4B-INSTRUCT.git
%cd NULLXES-SHINRA-4B-INSTRUCT
!pip install -q -e .

In [ ]:
import os
os.environ["PYTHONPATH"] = "/content/NULLXES-SHINRA-4B-INSTRUCT"
from tokenizer.special_tokens import ALL_SPECIAL_TOKENS, EOT, END_OF_TEXT, TOOL_CALL, DOCUMENT, CHAT_TEMPLATE, REASONING_POLICY
print("eot", EOT)
print("end_of_text", END_OF_TEXT)
print("tool_call", TOOL_CALL)
print("document", DOCUMENT)
print("reasoning_policy", REASONING_POLICY)
print("n_specials", len(ALL_SPECIAL_TOKENS))
assert "<|end|>" not in ALL_SPECIAL_TOKENS
assert "<|tool|>" not in ALL_SPECIAL_TOKENS
assert "{{- bos_token -}}" in CHAT_TEMPLATE
assert "<|eot|>" in CHAT_TEMPLATE
assert "tool_call" in CHAT_TEMPLATE

In [ ]:
!python -m architecture.param_count

## Tokenizer training

Положить representative mix (≥10B chars в проде) в `/content/tokmix`, затем:

```
!python -m tokenizer.train_tokenizer --input /content/tokmix --output-dir tokenizer/artifacts --vocab-size 131072
```

После freeze tokenizer + chat template больше не трогать.